# Liquidity Pool Integration Engine Demo

This notebook demonstrates the Liquidity Pool Integration Engine, which analyzes DEX rates and LP token yields across Tinyman, Algofi DEX, and other AMMs.

## Features Demonstrated:
- Tinyman pool yields and trading fees
- Algofi DEX liquidity mining rewards
- Impermanent loss analysis
- Multi-DEX yield aggregation
- Optimal LP allocation strategies


In [ ]:
# Setup and imports
import sys
import asyncio
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns

# Add the engine path
engine_path = Path.cwd().parent / "liquidity-pool-integration"
sys.path.insert(0, str(engine_path))

from core.tinyman_pools import TinymanPoolsAnalyzer
from core.algofi_pools import AlgofiPoolsAnalyzer
from core.pool_yield_calculator import PoolYieldCalculator

print("✅ Imports successful!")
print(f"Engine path: {engine_path}")

# Set style for plots
plt.style.use('default')
sns.set_palette("Set2")

## 1. Initialize DEX Pool Analyzers

In [ ]:
# Initialize the pool analyzers
tinyman_analyzer = TinymanPoolsAnalyzer()
algofi_analyzer = AlgofiPoolsAnalyzer()
pool_calculator = PoolYieldCalculator()

print("DEX pool analyzers initialized successfully!")

## 2. Tinyman Pools Analysis

In [ ]:
# Analyze Tinyman pools
tinyman_data = await tinyman_analyzer.analyze_tinyman_yields()

print("=== Tinyman Pools Analysis ===")
print(f"Weighted Average APY: {tinyman_data.weighted_average_apy:.2%}")
print(f"Total TVL: ${tinyman_data.total_tvl:,.0f}")
print(f"Total 24h Volume: ${tinyman_data.total_volume_24h:,.0f}")
print(f"Average Fee APY: {tinyman_data.average_fee_apy:.2%}")
print(f"Average Rewards APY: {tinyman_data.average_rewards_apy:.2%}")
print(f"Number of Pools: {len(tinyman_data.pools)}")
print(f"Confidence Score: {tinyman_data.confidence_score:.2f}")

print(f"\nTop Performing Pools: {', '.join(tinyman_data.top_performing_pools[:3])}")

# Display individual pools
print(f"\n--- Individual Pool Details ---")
for i, pool in enumerate(tinyman_data.pools[:5], 1):
    volume_to_liquidity = pool.volume_24h_usd / max(pool.liquidity_usd, 1)
    print(f"{i:2d}. {pool.asset_1_name}/{pool.asset_2_name:6s}: Total {pool.total_apy:.2%}, Fees {pool.apy_fees:.2%}, Rewards {pool.apy_rewards:.2%}")
    print(f"     Liquidity: ${pool.liquidity_usd:,.0f}, Volume: ${pool.volume_24h_usd:,.0f}, V/L: {volume_to_liquidity:.3f}")

## 3. Algofi DEX Pools Analysis

In [ ]:
# Analyze Algofi DEX pools
algofi_data = await algofi_analyzer.analyze_algofi_pools()

print("=== Algofi DEX Pools Analysis ===")
print(f"Weighted Average APY: {algofi_data.weighted_average_apy:.2%}")
print(f"Total TVL: ${algofi_data.total_tvl:,.0f}")
print(f"Total 24h Volume: ${algofi_data.total_volume_24h:,.0f}")
print(f"Average Trading APY: {algofi_data.average_trading_apy:.2%}")
print(f"Average Mining APY: {algofi_data.average_mining_apy:.2%}")
print(f"Average ALFI Rewards: {algofi_data.average_alfi_rewards:.2%}")
print(f"Total ALFI Emissions: {algofi_data.total_alfi_emissions:,.0f} ALFI/day")
print(f"Number of Pools: {len(algofi_data.pools)}")
print(f"Confidence Score: {algofi_data.confidence_score:.2f}")

# Display individual pools
print(f"\n--- Individual Pool Details ---")
for i, pool in enumerate(algofi_data.pools[:5], 1):
    print(f"{i:2d}. {pool.asset_1_name}/{pool.asset_2_name:6s}: Total {pool.total_apy:.2%}, Trading {pool.apy_trading_fees:.2%}, ALFI {pool.apy_alfi_rewards:.2%}")
    print(f"     Multiplier: {pool.multiplier_boost:.1f}x, Liquidity: ${pool.liquidity_usd:,.0f}, Volume: ${pool.volume_24h_usd:,.0f}")

## 4. Multi-DEX Pool Yield Aggregation

In [ ]:
# Calculate aggregated pool yields across all DEXes
aggregated_pools = await pool_calculator.calculate_all_pool_yields()

print("=== Multi-DEX Pool Yield Aggregation ===")
print(f"Overall Weighted Yield: {aggregated_pools.overall_weighted_yield:.2%}")
print(f"Total TVL Across DEXes: ${aggregated_pools.total_tvl_across_dexes:,.0f}")
print(f"Total 24h Volume: ${aggregated_pools.total_volume_24h:,.0f}")
print(f"Average IL Risk: {aggregated_pools.average_il_risk:.2%}")
print(f"Sustainability Score: {aggregated_pools.sustainability_score:.2f}")
print(f"Number of Pool Pairs: {len(aggregated_pools.pool_summaries)}")

# DEX market shares
print(f"\n--- DEX Market Shares (by TVL) ---")
for dex, share in aggregated_pools.dex_market_shares.items():
    tvl_share = share * aggregated_pools.total_tvl_across_dexes
    print(f"{dex:15s}: {share:.1%} (${tvl_share:,.0f})")

# Pool summaries
print(f"\n--- Pool Pair Summaries ---")
for summary in aggregated_pools.pool_summaries[:5]:
    total_liquidity = sum(summary.liquidity_values.values())
    print(f"{summary.asset_pair:12s}: Avg {summary.weighted_average_yield:.2%}, Best {summary.best_yield_dex}, Spread {summary.yield_spread:.2%}")
    print(f"                  Liquidity: ${total_liquidity:,.0f}, DEXes: {len(summary.dex_yields)}")

## 5. Impermanent Loss Analysis

In [ ]:
# Analyze impermanent loss for specific pools
print("=== Impermanent Loss Analysis ===")

# Get ALGO/USDC pool from Tinyman for IL analysis
algo_usdc_pool = await tinyman_analyzer.get_pool_by_pair('ALGO', 'USDC')

if algo_usdc_pool:
    il_analysis = await tinyman_analyzer.analyze_impermanent_loss(algo_usdc_pool.pool_id)
    
    print(f"\nALGO/USDC Pool (Tinyman):")
    print(f"  Current IL: {il_analysis.current_il_percentage:.2%}")
    print(f"  Risk Category: {il_analysis.risk_category}")
    print(f"  Break-even Days: {il_analysis.break_even_days:.0f}")
    print(f"  Hedging Cost: {il_analysis.hedging_cost:.2%}")
    print(f"  Net APY after IL Risk: {il_analysis.net_apy_after_il_risk:.2%}")
    
    print(f"\n  IL at Different Price Changes:")
    for price_change, il_value in il_analysis.il_at_price_changes.items():
        print(f"    {price_change:6s}: {il_value:.2%}")
else:
    print("ALGO/USDC pool not found in Tinyman data")

# Get top yield opportunities with risk analysis
print(f"\n--- Top Yield Opportunities (Risk-Adjusted) ---")
opportunities = await tinyman_analyzer.get_top_yield_opportunities(100000)

for i, opp in enumerate(opportunities[:5], 1):
    print(f"{i:2d}. {opp['pair']:12s}: {opp['total_apy']:.2%} APY, Risk: {opp['il_risk_category']}, Score: {opp['risk_adjusted_score']:.2%}")
    print(f"     Break-even: {opp['break_even_days']:.0f} days, Liquidity: ${opp['liquidity_usd']:,.0f}")

## 6. ALFI Emissions Impact Analysis

In [ ]:
# Analyze ALFI token emissions impact
emissions_analysis = await algofi_analyzer.analyze_alfi_emissions_impact()

if "error" not in emissions_analysis:
    print("=== ALFI Emissions Impact Analysis ===")
    print(f"Total Daily ALFI Emissions: {emissions_analysis['total_daily_alfi_emissions']:,.0f}")
    print(f"Total Daily USD Value: ${emissions_analysis['total_daily_usd_value']:,.0f}")
    print(f"ALFI Price: ${emissions_analysis['alfi_price_usd']:.3f}")
    print(f"Average ALFI APY: {emissions_analysis['average_alfi_apy']:.2%}")
    print(f"ALFI Contribution to Total Yield: {emissions_analysis['weighted_alfi_contribution']:.1%}")
    print(f"Sustainability Score: {emissions_analysis['sustainability_score']:.2f}")
    
    print(f"\n--- High Multiplier Pools ---")
    high_mult_pools = emissions_analysis['high_multiplier_pools']
    for pool in high_mult_pools[:3]:
        print(f"  {pool['pair']:12s}: {pool['alfi_apy']:.2%} ALFI APY, {pool['multiplier']:.1f}x multiplier")
        
    print(f"\n--- Emissions Distribution (Top 5) ---")
    for pool in emissions_analysis['emissions_by_pool'][:5]:
        print(f"  {pool['pair']:12s}: {pool['alfi_apy']:.2%} APY, {pool['emissions_share']:.1%} of total emissions")
else:
    print(f"ALFI emissions analysis error: {emissions_analysis['error']}")

## 7. Optimal LP Allocation Strategy

In [ ]:
# Calculate optimal LP allocation strategies
print("=== Optimal LP Allocation Strategies ===")

# Test different risk tolerances
risk_levels = ['low', 'medium', 'high']
target_apy = 0.08  # 8% target APY

for risk_level in risk_levels:
    allocation = await pool_calculator.get_optimal_lp_allocation(target_apy, risk_level)
    
    if "error" not in allocation:
        print(f"\n{risk_level.upper()} Risk Tolerance:")
        print(f"  Target APY: {allocation['target_apy']:.2%}")
        print(f"  Expected APY: {allocation['expected_apy']:.2%}")
        print(f"  Expected IL Risk: {allocation['expected_il_risk']:.2f}")
        print(f"  Diversification Score: {allocation['diversification_score']:.2f}")
        print(f"  Opportunities Considered: {allocation['total_opportunities_considered']}")
        
        print(f"  Top Allocations:")
        for alloc in allocation['allocations'][:3]:
            print(f"    {alloc['pair']:12s} on {alloc['dex']:8s}: {alloc['weight']:.1%} ({alloc['total_apy']:.2%} APY)")
    else:
        print(f"\n{risk_level.upper()} Risk: {allocation['error']}")

## 8. Yield Projections

In [ ]:
# Project yields for the next 30 days
projections = await pool_calculator.project_yields(30)

print("=== 30-Day Yield Projections ===")
print(f"Generated {len(projections)} projections")

# Show top 5 projected yields
sorted_projections = sorted(projections, key=lambda x: x.projected_30d_yield, reverse=True)

print(f"\n--- Top Projected Yields ---")
for i, proj in enumerate(sorted_projections[:5], 1):
    print(f"{i:2d}. {proj.pool_pair:12s} on {proj.dex_name:8s}:")
    print(f"     7d: {proj.projected_7d_yield:.2%}, 30d: {proj.projected_30d_yield:.2%}, 90d: {proj.projected_90d_yield:.2%}")
    print(f"     IL Estimate: {proj.impermanent_loss_estimate:.2%}, Net After IL: {proj.net_yield_after_il:.2%}")
    print(f"     Risk Factors: {len(proj.risk_factors)} ({', '.join(proj.risk_factors[:2])}...)")
    
    # Show confidence intervals
    ci_68 = proj.confidence_intervals.get('68%', (0, 0))
    print(f"     68% CI: {ci_68[0]:.2%} - {ci_68[1]:.2%}")

## 9. Visualization: Multi-DEX Comparison

In [ ]:
# Create comprehensive multi-DEX visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. DEX Market Share by TVL
dex_names = list(aggregated_pools.dex_market_shares.keys())
market_shares = list(aggregated_pools.dex_market_shares.values())
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFD93D', '#A8E6CF']

ax1.pie(market_shares, labels=dex_names, autopct='%1.1f%%', startangle=90, colors=colors[:len(dex_names)])
ax1.set_title(f'DEX Market Share by TVL\nTotal: ${aggregated_pools.total_tvl_across_dexes:,.0f}', fontweight='bold')

# 2. Yield Comparison by Asset Pair
pair_data = []
for summary in aggregated_pools.pool_summaries[:5]:  # Top 5 pairs
    for dex, yield_val in summary.dex_yields.items():
        pair_data.append({
            'Pair': summary.asset_pair,
            'DEX': dex,
            'Yield': yield_val * 100
        })

if pair_data:
    df_pairs = pd.DataFrame(pair_data)
    pair_pivot = df_pairs.pivot(index='Pair', columns='DEX', values='Yield')
    pair_pivot.plot(kind='bar', ax=ax2, color=colors[:len(pair_pivot.columns)])
    ax2.set_ylabel('Yield (%)', fontweight='bold')
    ax2.set_title('Yields by Asset Pair and DEX', fontweight='bold')
    ax2.legend(title='DEX')
    ax2.grid(axis='y', alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)

# 3. Risk vs Yield Scatter (IL Risk vs APY)
risk_yield_data = []
for opp in aggregated_pools.top_yield_opportunities[:10]:
    risk_yield_data.append({
        'risk': opp.get('risk_score', 0.5) * 100,
        'yield': opp['total_apy'] * 100,
        'size': opp['liquidity_usd'] / 50000,  # Scale for bubble size
        'label': f"{opp['pair']} ({opp['dex']})"
    })

if risk_yield_data:
    risks = [d['risk'] for d in risk_yield_data]
    yields = [d['yield'] for d in risk_yield_data]
    sizes = [d['size'] for d in risk_yield_data]
    
    scatter = ax3.scatter(risks, yields, s=sizes, alpha=0.6, c=range(len(risks)), cmap='viridis')
    ax3.set_xlabel('Risk Score', fontweight='bold')
    ax3.set_ylabel('Yield (%)', fontweight='bold')
    ax3.set_title('Risk vs Yield (Bubble size = Liquidity)', fontweight='bold')
    ax3.grid(alpha=0.3)
    
    # Add labels for top opportunities
    for i, d in enumerate(risk_yield_data[:5]):
        ax3.annotate(d['label'].split('(')[0], (d['risk'], d['yield']), 
                    xytext=(2, 2), textcoords='offset points', fontsize=8)

# 4. Volume vs Liquidity Efficiency
efficiency_data = []
for summary in aggregated_pools.pool_summaries:
    total_volume = sum(summary.volume_24h_values.values())
    total_liquidity = sum(summary.liquidity_values.values())
    if total_liquidity > 0:
        efficiency = total_volume / total_liquidity
        efficiency_data.append({
            'pair': summary.asset_pair,
            'efficiency': efficiency,
            'yield': summary.weighted_average_yield * 100,
            'liquidity': total_liquidity
        })

if efficiency_data:
    efficiencies = [d['efficiency'] for d in efficiency_data]
    yields_eff = [d['yield'] for d in efficiency_data]
    liquidities = [d['liquidity'] / 100000 for d in efficiency_data]  # Scale for bubble size
    
    scatter2 = ax4.scatter(efficiencies, yields_eff, s=liquidities, alpha=0.6, c=range(len(efficiencies)), cmap='plasma')
    ax4.set_xlabel('Volume/Liquidity Ratio', fontweight='bold')
    ax4.set_ylabel('Weighted Yield (%)', fontweight='bold')
    ax4.set_title('Trading Efficiency vs Yield', fontweight='bold')
    ax4.grid(alpha=0.3)
    
    # Add labels
    for i, d in enumerate(efficiency_data[:5]):
        ax4.annotate(d['pair'], (d['efficiency'], d['yield']), 
                    xytext=(2, 2), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nVisualization Summary:")
print(f"Total DEXes: {len(dex_names)}")
print(f"Total Asset Pairs: {len(aggregated_pools.pool_summaries)}")
print(f"Top Opportunities: {len(aggregated_pools.top_yield_opportunities)}")

## 10. Advanced Analytics: Pool Efficiency & Sustainability

In [ ]:
# Analyze pool efficiency across DEXes
algofi_efficiency = await algofi_analyzer.compare_pool_efficiency()

print("=== Advanced Pool Analytics ===")

if "error" not in algofi_efficiency:
    print(f"\nAlgofi Pool Efficiency Analysis:")
    
    most_efficient = algofi_efficiency['most_efficient_pool']
    least_efficient = algofi_efficiency['least_efficient_pool']
    
    if most_efficient:
        print(f"  Most Efficient: {most_efficient['pair']} (Score: {most_efficient['overall_score']:.2f})")
        print(f"    Volume/Liquidity: {most_efficient['volume_to_liquidity_ratio']:.3f}")
        print(f"    Capital Efficiency: {most_efficient['capital_efficiency']:.2f}")
    
    if least_efficient:
        print(f"  Least Efficient: {least_efficient['pair']} (Score: {least_efficient['overall_score']:.2f})")
    
    print(f"  Average Efficiency Score: {algofi_efficiency['average_efficiency']:.2f}")
    
    print(f"\n  Top 3 Most Efficient Pools:")
    for i, pool in enumerate(algofi_efficiency['pools'][:3], 1):
        print(f"    {i}. {pool['pair']:12s}: Score {pool['overall_score']:.2f}, APY {pool['total_apy']:.2%}")

# Calculate sustainability metrics
print(f"\nSustainability Analysis:")
print(f"  Overall Sustainability Score: {aggregated_pools.sustainability_score:.2f}")

# Calculate fee vs rewards ratio for sustainability
total_fee_yield = 0
total_reward_yield = 0
total_weight = 0

for summary in aggregated_pools.pool_summaries:
    for dex, liquidity in summary.liquidity_values.items():
        trading_fee_yield = summary.trading_fee_yields.get(dex, 0)
        reward_yield = summary.reward_yields.get(dex, 0)
        
        total_fee_yield += trading_fee_yield * liquidity
        total_reward_yield += reward_yield * liquidity
        total_weight += liquidity

if total_weight > 0:
    avg_fee_yield = total_fee_yield / total_weight
    avg_reward_yield = total_reward_yield / total_weight
    fee_to_reward_ratio = avg_fee_yield / max(avg_reward_yield, 0.001)
    
    print(f"  Average Fee Yield: {avg_fee_yield:.2%}")
    print(f"  Average Reward Yield: {avg_reward_yield:.2%}")
    print(f"  Fee/Reward Ratio: {fee_to_reward_ratio:.2f} (higher = more sustainable)")

# Risk distribution analysis
il_risks = []
for summary in aggregated_pools.pool_summaries:
    # Estimate IL risk based on asset pair volatility
    if 'goBTC' in summary.asset_pair or 'goETH' in summary.asset_pair:
        il_risk = 0.8  # High IL risk
    elif 'USDC/USDT' in summary.asset_pair:
        il_risk = 0.1  # Low IL risk
    else:
        il_risk = 0.5  # Medium IL risk
    il_risks.append(il_risk)

if il_risks:
    avg_il_risk = sum(il_risks) / len(il_risks)
    print(f"  Average IL Risk: {avg_il_risk:.2%}")
    
    risk_categories = {'Low': 0, 'Medium': 0, 'High': 0}
    for risk in il_risks:
        if risk < 0.3:
            risk_categories['Low'] += 1
        elif risk < 0.7:
            risk_categories['Medium'] += 1
        else:
            risk_categories['High'] += 1
    
    print(f"  Risk Distribution: Low: {risk_categories['Low']}, Medium: {risk_categories['Medium']}, High: {risk_categories['High']}")

## 11. Summary and Key Insights

In [ ]:
# Create comprehensive summary
print("=== LIQUIDITY POOL INTEGRATION ENGINE SUMMARY ===")
print("="*60)

print(f"\n📊 AGGREGATED METRICS:")
print(f"   • Overall Weighted Yield: {aggregated_pools.overall_weighted_yield:.2%}")
print(f"   • Total TVL Across DEXes: ${aggregated_pools.total_tvl_across_dexes:,.0f}")
print(f"   • Total 24h Volume: ${aggregated_pools.total_volume_24h:,.0f}")
print(f"   • Average IL Risk: {aggregated_pools.average_il_risk:.2%}")
print(f"   • Sustainability Score: {aggregated_pools.sustainability_score:.2f}")

print(f"\n🏦 DEX BREAKDOWN:")
for dex, share in aggregated_pools.dex_market_shares.items():
    tvl_share = share * aggregated_pools.total_tvl_across_dexes
    print(f"   • {dex:15s}: {share:.1%} market share (${tvl_share:,.0f})")

print(f"\n🎯 TOP YIELD OPPORTUNITIES:")
for i, opp in enumerate(aggregated_pools.top_yield_opportunities[:5], 1):
    print(f"   {i}. {opp['pair']:12s} on {opp['dex']:8s}: {opp['total_apy']:.2%} APY")
    print(f"      Liquidity: ${opp['liquidity_usd']:,.0f}, Risk Score: {opp.get('risk_score', 0.5):.2f}")

print(f"\n🔍 PROTOCOL ANALYSIS:")
print(f"   • Tinyman Pools: {len(tinyman_data.pools)}, Avg APY: {tinyman_data.weighted_average_apy:.2%}")
print(f"   • Algofi Pools: {len(algofi_data.pools)}, Avg APY: {algofi_data.weighted_average_apy:.2%}")
print(f"   • Total Asset Pairs: {len(aggregated_pools.pool_summaries)}")

if "error" not in emissions_analysis:
    print(f"\n💰 ALFI EMISSIONS IMPACT:")
    print(f"   • Daily ALFI Emissions: {emissions_analysis['total_daily_alfi_emissions']:,.0f}")
    print(f"   • Daily USD Value: ${emissions_analysis['total_daily_usd_value']:,.0f}")
    print(f"   • Avg ALFI APY: {emissions_analysis['average_alfi_apy']:.2%}")
    print(f"   • Sustainability Score: {emissions_analysis['sustainability_score']:.2f}")

if il_analysis:
    print(f"\n⚠️ IMPERMANENT LOSS INSIGHTS:")
    print(f"   • ALGO/USDC IL Risk: {il_analysis.risk_category}")
    print(f"   • Break-even Period: {il_analysis.break_even_days:.0f} days")
    print(f"   • Hedging Cost: {il_analysis.hedging_cost:.2%}")
    print(f"   • Net APY After IL: {il_analysis.net_apy_after_il_risk:.2%}")

if len(sorted_projections) > 0:
    print(f"\n📈 YIELD PROJECTIONS (30 days):")
    top_proj = sorted_projections[0]
    print(f"   • Best Projected: {top_proj.pool_pair} on {top_proj.dex_name} ({top_proj.projected_30d_yield:.2%})")
    print(f"   • Net After IL: {top_proj.net_yield_after_il:.2%}")
    print(f"   • Risk Factors: {len(top_proj.risk_factors)}")

print(f"\n🎯 OPTIMAL ALLOCATION (Medium Risk):")
medium_allocation = await pool_calculator.get_optimal_lp_allocation(0.08, "medium")
if "error" not in medium_allocation:
    print(f"   • Expected APY: {medium_allocation['expected_apy']:.2%}")
    print(f"   • Expected IL Risk: {medium_allocation['expected_il_risk']:.2f}")
    print(f"   • Diversification Score: {medium_allocation['diversification_score']:.2f}")
    print(f"   • Top Allocation: {medium_allocation['allocations'][0]['pair']} ({medium_allocation['allocations'][0]['weight']:.1%})")

print(f"\n✅ ENGINE STATUS:")
print(f"   • DEXes Analyzed: {len(aggregated_pools.dex_market_shares)}")
print(f"   • Pool Pairs Tracked: {len(aggregated_pools.pool_summaries)}")
print(f"   • Data Quality: {'High' if aggregated_pools.sustainability_score > 0.7 else 'Medium'}")
print(f"   • IL Analysis: {'Enabled' if il_analysis else 'Limited'}")
print(f"   • Last Updated: {aggregated_pools.timestamp.strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*60)
print("🎉 Liquidity Pool Integration Engine Demo Complete!")